# UCCI quickstart

[UCCI](https://arxiv.org/abs/2605.18796) routes each query to a small or a large LLM. It turns the small model's token probabilities into a calibrated probability that its answer is wrong, and calls the large model only when that probability is above a threshold chosen to hit an accuracy target at the lowest cost.

This notebook fits a router, checks its calibration, computes the signal from a real model, and saves the router for the CLI. Code: [github.com/varunkotte6/ucci](https://github.com/varunkotte6/ucci).

In [ ]:
!pip install -q ucci-router

## 1. Fit a router on logged traffic

For each logged query you need the small model's u(x) and whether each model was right. The data here is simulated; swap in your own logs.

In [ ]:
import numpy as np
from ucci import UCCIRouter

rng = np.random.default_rng(0)
n = 10_000
u = rng.beta(2, 5, n)                                                  # small-model uncertainty u(x)
small_ok = (rng.random(n) >= 1 / (1 + np.exp(-10 * (u - 0.45)))).astype(float)
large_ok = (rng.random(n) < 0.95).astype(float)
cal, val, test = np.split(rng.permutation(n), [3_000, 5_000])       # 30% / 20% / 50%

router = UCCIRouter(c_small=1.0, c_large=3.0)
router.calibrate(u[cal], 1 - small_ok[cal])                            # fit g on calibration
choice = router.choose_threshold(u[val], small_ok[val], large_ok[val], tau=0.90)
result = router.evaluate(u[test], small_ok[test], large_ok[test])

print(f"theta = {choice.theta}")
print(f"test accuracy {result.accuracy:.3f} at cost {result.cost:.2f} "
      f"(large model alone: {large_ok[test].mean():.3f} at 3.00)")

## 2. Check the calibration

Raw u(x) is not a probability. After the isotonic fit, a forecast of 0.3 means the small model is wrong about 30% of the time.

In [ ]:
from ucci import ece, plotting

p_hat = router.error_probability(u[test])
wrong = 1 - small_ok[test]
print(f"ECE raw u: {ece(u[test], wrong):.3f}   ECE calibrated: {ece(p_hat, wrong):.3f}")
plotting.reliability_diagram({"raw u(x)": u[test], "calibrated": p_hat}, wrong, title="Test split");

## 3. u(x) from a real model

`generate_with_signals` runs greedy decoding and returns u(x) for every prompt. A 0.5B model runs fine on a free Colab CPU. Answers the model is less sure of get a higher u(x).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from ucci.integrations.transformers import generate_with_signals

name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

questions = [
    "What is the capital of France? Answer in one word.",
    "Who wrote the novel 'The Master and Margarita'? Answer with the name only.",
    "What is 17 * 23? Answer with the number only.",
]
prompts = [
    tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    for q in questions
]
out = generate_with_signals(model, tokenizer, prompts, max_new_tokens=32)

for text, u_q in zip(out.texts, out.u):
    print(f"u(x) = {u_q:.3f}  | {text.strip()}")

To route real traffic, log u(x) and correctness for your own small and large models and fit as in step 1.

## 4. Save the router

The JSON file is read by `UCCIRouter.load`, the `ucci` command line tool and the Rust crate.

In [ ]:
router.save("router.json")
!ucci route --router router.json --u 0.02 0.10 0.35

## Cite

```bibtex
@article{kotte2026ucci,
  title   = {{UCCI}: Calibrated Uncertainty for Cost-Optimal {LLM} Cascade Routing},
  author  = {Kotte, Varun},
  journal = {arXiv preprint arXiv:2605.18796},
  year    = {2026}
}
```